In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
import matplotlib.patches as patches
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

# 1. 64괘 대체 코드 생성 (AA ~ HH)
chars = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
code_names = [f"{chars[i]}{chars[j]}" for i in range(8) for j in range(8)]
code_dict = {i: code_names[i] for i in range(64)}

def get_full_info(val_6bit):
    return {
        'code': code_dict.get(val_6bit, "??"),
        'bin': format(val_6bit, '06b'),
        'dec': val_6bit,
        'oct': format(val_6bit, '02o'),
        'hex': format(val_6bit, '02X')
    }

# 전역 변수로 애니메이션 객체 유지 (사라짐 방지)
anim_store = None

def create_fixed_simulation(start_angle, end_angle):
    global anim_store
    fig, ax = plt.subplots(figsize=(10, 6))
    plt.subplots_adjust(right=0.5)

    def update(frame):
        ax.clear()
        ax.set_xlim(-1.2, 4.0); ax.set_ylim(-1.2, 1.2)
        ax.set_aspect('equal'); ax.axis('off')

        current_angle_deg = start_angle + (end_angle - start_angle) * (frame / 100)
        angle_rad = np.radians(current_angle_deg)
        val_total = int(((current_angle_deg % 360) / 360) * (2**18))
        
        layers = [get_full_info((val_total >> shift) & 0x3F) for shift in [12, 6, 0]]
        labels = ["L1 (Major)", "L2 (Middle)", "L3 (Minor)"]

        # 시각화 요소
        ax.add_patch(patches.Wedge((0,0), 1.0, 0, current_angle_deg % 360, color='skyblue', alpha=0.2))
        ax.plot([0, np.cos(angle_rad)], [0, np.sin(angle_rad)], color='#34495e', marker='o', lw=4, ms=8)
        ax.add_patch(patches.Circle((0,0), 0.05, color='#e74c3c', zorder=5))
        ax.add_patch(patches.Circle((0,0), 1.0, fill=False, color='#bdc3c7', ls='--'))
        
        # 데이터 테이블 텍스트
        info_str = (
            f"Angle: {current_angle_deg:.1f}°\n"
            f"Total Pulse: {val_total:,}\n"
            f"{'='*38}\n"
            f"{'Layer':<12} | {'Code':<4} | {'Bin':<6} | {'Dec':<3} | {'Oct':<3} | {'Hex':<3}\n"
            f"{'-'*38}\n"
        )
        for label, data in zip(labels, layers):
            info_str += (f"{label:<12} | {data['code']:<4} | {data['bin']} | "
                         f"{data['dec']:<3} | {data['oct']:<3} | {data['hex']:<3}\n")

        ax.text(1.3, 0.9, info_str, fontsize=9, family='monospace', va='top', 
                bbox=dict(facecolor='white', edgecolor='#34495e', boxstyle='round,pad=0.5'))
        ax.set_title("18-bit Multi-Base Fixed Simulation", fontsize=12, fontweight='bold')

    # 애니메이션 생성 및 HTML 변환
    anim_store = FuncAnimation(fig, update, frames=101, interval=60, blit=False)
    js_html = HTML(anim_store.to_jshtml())
    plt.close(fig) # 별도의 정지된 팝업창 생성을 방지
    return js_html

# 3. UI 구성
style = {'description_width': '100px'}
start_slider = widgets.IntSlider(value=0, min=0, max=360, description='시작 각도:', style=style)
end_slider = widgets.IntSlider(value=360, min=0, max=360, description='종료 각도:', style=style)
run_btn = widgets.Button(description="시뮬레이션 업데이트", button_style='info', icon='refresh')
# 결과가 출력될 고정 영역
output_area = widgets.Output()

def on_click_refresh(b):
    with output_area:
        clear_output(wait=True) # 이전 애니메이션 삭제
        display(create_fixed_simulation(start_slider.value, end_slider.value))

run_btn.on_click(on_click_refresh)

# 화면 배치: 슬라이더 아래에 바로 출력 영역 배치
layout = widgets.VBox([
    widgets.HTML("<h2>System Control Panel</h2>"),
    widgets.HBox([start_slider, end_slider]),
    run_btn,
    widgets.HTML("<br>"),
    output_area # 이 영역에 애니메이션이 고정됨
])

display(layout)

# 초기 실행 시 한 번 보여주기
with output_area:
    display(create_fixed_simulation(0, 360))